In [1]:
import torch
from torch import nn
from d2l import torch as d2l

In [2]:
def NiN_block(in_channel, out_channel, kernel_size, strides, padding):
    return nn.Sequential(
        nn.Conv2d(in_channel, out_channel, kernel_size, strides, padding),
        nn.ReLU(),
        nn.Conv2d(out_channel, out_channel, kernel_size = 1),
        nn.ReLU(),
        nn.Conv2d(out_channel, out_channel, kernel_size = 1),
        nn.ReLU()
    )

In [3]:
def NiN(conv_array):
    blocks = []
    in_channel = 1
    cout = 0
    for (out_channel, kernel_size, strides, padding) in conv_array:
        cout += 1
        blocks.append(NiN_block(in_channel, out_channel, kernel_size, strides, padding))
        in_channel = out_channel
        if cout == len(conv_array):
            blocks.append(nn.AdaptiveAvgPool2d((1, 1)))
        else:
            blocks.append(nn.MaxPool2d(kernel_size=3, stride=2))
    return nn.Sequential(*blocks, nn.Flatten())

In [6]:
conv_array = ((96, 11, 4, 0), (256, 5, 1, 2), (384, 3, 1, 1), (10, 3, 1, 1))
net = (NiN(conv_array))

In [ ]:
lr, num_epochs, batch_size = 0.1, 10, 128
train_iter, test_iter = d2l.load_data_fashion_mnist(batch_size, resize=224)
d2l.train_ch6(net, train_iter, test_iter, num_epochs, lr, d2l.try_gpu())